In [10]:
import pandas as pd

df = pd.read_csv('netflix_titles.csv')

# 1. title, country \uc678 \uce7c\ub7fc \uc0ad\uc81c, country\uac00 nan\uc778 \ud589 \uc0ad\uc81c
df = df[['title', 'country']]
df = df.dropna(subset=['country'])

# \uad6d\uac00 \ubaa9\ub85d\uc73c\ub85c \ubd84\ub9ac \ud6c4 \ucc38\uc5ec \uad6d\uac00 \uc218 \uacc4\uc0b0
df['country'] = df['country'].str.split(',').apply(lambda countries: [c.strip() for c in countries])

# 2, 3, 5. co_production \uce7c\ub7fc \uc0dd\uc131 (\ucc38\uc5ec \uad6d\uac00 \uc218)
df['co_production'] = df['country'].apply(len)

# 4. \uad6d\uac00 \uc218\ub9cc\ud07c \ud589 \ubd84\ub9ac
df = df.explode('country')

# 6. \uacf5\ub3d9\uc81c\uc791 \ud589\uc740 \uad6d\uac00\uba85 \ub4a4\uc5d0 _co_production \ucd94\uac00
is_co_production = df['co_production'] > 1
df.loc[is_co_production, 'country'] = df.loc[is_co_production, 'country'] + '_co_production'

df = df.reset_index(drop=True)

df

,title,country,co_production
0,3%,Brazil,1
1,7:19,Mexico,1
2,23:59,Singapore,1
3,9,United States,1
4,21,United States,1
...,...,...,...
9062,Zubaan,India,1
9063,Zumbo's Just Desserts,Australia,1
9064,ZZ TOP: THAT LITTLE OL' BAND FROM TEXAS,United Kingdom_co_production,3
9065,ZZ TOP: THAT LITTLE OL' BAND FROM TEXAS,Canada_co_production,3


In [11]:
df['country_clean'] = df['country'].str.replace('_co_production', '', regex=False)

# 1. 단독제작의 경우만 인정
rank_solo_only = (
    df[df['co_production'] == 1]
    .groupby('country_clean')
    .size()
    .sort_values(ascending=False)
)

rank_solo_only

country_clean
United States     2555
India              923
United Kingdom     397
Japan              226
South Korea        183
                  ... 
Namibia              1
Senegal              1
Venezuela            1
West Germany         1
Zimbabwe             1
Length: 69, dtype: int64

In [12]:
# 2. 공동제작도 하나의 컨텐츠 제작으로 인정
rank_full_credit = (
    df
    .groupby('country_clean')
    .size()
    .sort_values(ascending=False)
)

rank_full_credit

country_clean
United States     3297
India              990
United Kingdom     723
Canada             412
France             349
                  ... 
Slovakia             1
Samoa                1
Syria                1
Uganda               1
Vatican City         1
Length: 118, dtype: int64

In [13]:
# 3. 공동제작의 경우 1 / co_production 만큼 인정 (예: 2개국 공동제작이면 국가당 0.5)
rank_fractional = (
    df.assign(weight=1 / df['co_production'])
    .groupby('country_clean')['weight']
    .sum()
    .sort_values(ascending=False)
)

rank_fractional

country_clean
United States     2868.939286
India              951.602381
United Kingdom     523.546429
Canada             271.885714
Japan              249.509524
                     ...     
Somalia              0.200000
Vatican City         0.200000
Syria                0.200000
Armenia              0.083333
Mongolia             0.083333
Name: weight, Length: 118, dtype: float64